# 03 — Feature Engineering

This notebook transforms the molecular structures from the AqSolDB benchmark into numerical representations suitable for machine learning.

The following molecular representations are generated:

1. Basic physicochemical descriptors
2. MACCS structural keys
3. Morgan fingerprints
4. Physicochemical descriptors + MACCS
5. Physicochemical descriptors + Morgan fingerprints

The same transformations are applied independently to the official TDC `train_val` and `test` partitions.

No model-dependent preprocessing, feature selection, scaling or train/validation splitting is performed in this notebook. These steps belong to the modelling pipeline and must be fitted using training data only.

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator
RDLogger.DisableLog("rdApp.warning")

from tdc.benchmark_group import admet_group

In [11]:
BENCHMARK_NAME = "Solubility_AqSolDB"

MORGAN_RADIUS = 2
MORGAN_FP_SIZE = 2048

PROCESSED_DIR = Path("../data/processed/aqsoldb")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load benchmark

The official TDC benchmark partitions are loaded without modifying the original split.

Feature engineering is applied separately to `train_val` and `test` to preserve the benchmark structure.

In [12]:
group = admet_group(path="../data/raw/")

benchmark = group.get(BENCHMARK_NAME)

train_val = benchmark["train_val"].copy()
test = benchmark["test"].copy()

print("Benchmark:", benchmark["name"])
print("Train/validation:", train_val.shape)
print("Test:", test.shape)

Found local copy...


Benchmark: solubility_aqsoldb
Train/validation: (7985, 3)
Test: (1997, 3)


## 2. Molecular parsing

SMILES strings are converted into RDKit molecular objects.

Molecular validity was already assessed during exploratory data analysis, where all training/validation molecules were successfully parsed by RDKit.

The RDKit molecular objects created here are temporary and will not be persisted to disk.

In [13]:
train_val["ROMol"] = train_val["Drug"].apply(Chem.MolFromSmiles)
test["ROMol"] = test["Drug"].apply(Chem.MolFromSmiles)

## 3. Basic physicochemical descriptors

A small set of interpretable molecular descriptors is used to represent global physicochemical properties.

The selected descriptors include the classical Lipinski Rule-of-Five properties together with additional descriptors related to molecular size, polarity and flexibility.

Features:

- Molecular weight
- LogP
- Hydrogen-bond donors
- Hydrogen-bond acceptors

In addition, we will use ESOL (https://pubs.acs.org/jcisd8/article-abstract/44/3/1000/3504546/ESOL-Estimating-Aqueous-Solubility-Directly-from?redirectedFrom=PDF) descriptors, consisting of:

- Molecular weight
- LogP
- Rotable bonds
- Aromatic Proportion

Finally, we will use an extended version of descriptors combaining Rule-of-five, ESOL article and other descriptors related to molecular size, plarity and flexibility:

- Molecular weight
- LogP
- Rotable bonds
- Aromatic Proportion
- Hydrogen-bond donors
- Hydrogen-bond acceptors
- Topological polar surface area (TPSA)
- Rotatable bonds
- Ring count
- Heavy atom count


These features provide a compact and chemically interpretable baseline representation.

In [14]:
def compute_RO5_descriptors(mol):
    return pd.Series(
        {
            "MolWt": Descriptors.MolWt(mol),
            "LogP": Descriptors.MolLogP(mol),
            "HBD": Descriptors.NumHDonors(mol),
            "HBA": Descriptors.NumHAcceptors(mol),
        }
    )

In [15]:
def aromatic_proportion(mol):
    heavy_atoms = Descriptors.HeavyAtomCount(mol)
    if heavy_atoms == 0:    
        return 0.0

    aromatic_atoms = sum(
        atom.GetIsAromatic()
        for atom in mol.GetAtoms()
    )

    return aromatic_atoms / heavy_atoms

In [16]:
def compute_esol_descriptors(mol):
    return pd.Series({
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "RotatableBonds": Descriptors.NumRotatableBonds(mol),
        "AromaticProportion": aromatic_proportion(mol),
    })

In [17]:
def compute_extended_descriptors(mol):
    return pd.Series(
        {
            "MolWt": Descriptors.MolWt(mol),
            "LogP": Descriptors.MolLogP(mol),
            "HBD": Descriptors.NumHDonors(mol),
            "HBA": Descriptors.NumHAcceptors(mol),
            "TPSA": Descriptors.TPSA(mol),
            "RotatableBonds": Descriptors.NumRotatableBonds(mol),
            "RingCount": Descriptors.RingCount(mol),
            "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
            "AromaticProportion": aromatic_proportion(mol),
        }
    )

In [18]:
X_train_val_RO5_descriptors = (
    train_val["ROMol"]
    .apply(compute_RO5_descriptors)
    .reset_index(drop=True)
)

X_test_RO5_descriptors = (
    test["ROMol"]
    .apply(compute_RO5_descriptors)
    .reset_index(drop=True)
)

In [19]:
X_train_val_RO5_descriptors.head()

,MolWt,LogP,HBD,HBA
0,140.569,2.15250,0.0,1.0
1,118.179,2.63802,0.0,0.0
2,149.193,1.56510,0.0,2.0
3,192.258,2.57080,0.0,2.0
4,260.255,3.70480,0.0,2.0


In [20]:
X_train_val_esol_descriptors = (
    train_val["ROMol"]
    .apply(compute_esol_descriptors)
    .reset_index(drop=True)
)

X_test_esol_descriptors = (
    test["ROMol"]
    .apply(compute_esol_descriptors)
    .reset_index(drop=True)
)

In [21]:
X_train_val_esol_descriptors.head()

,MolWt,LogP,RotatableBonds,AromaticProportion
0,140.569,2.15250,1.0,0.666667
1,118.179,2.63802,1.0,0.666667
2,149.193,1.56510,2.0,0.545455
3,192.258,2.57080,3.0,0.428571
4,260.255,3.70480,6.0,0.333333


In [22]:
X_train_val_extended_descriptors = (
    train_val["ROMol"]
    .apply(compute_extended_descriptors)
    .reset_index(drop=True)
)

X_test_extended_descriptors = (
    test["ROMol"]
    .apply(compute_extended_descriptors)
    .reset_index(drop=True)
)

In [23]:
X_train_val_extended_descriptors.head()

,MolWt,LogP,HBD,HBA,TPSA,RotatableBonds,RingCount,HeavyAtomCount,AromaticProportion
0,140.569,2.15250,0.0,1.0,17.07,1.0,1.0,9.0,0.666667
1,118.179,2.63802,0.0,0.0,0.00,1.0,1.0,9.0,0.666667
2,149.193,1.56510,0.0,2.0,20.31,2.0,1.0,11.0,0.545455
3,192.258,2.57080,0.0,2.0,26.30,3.0,1.0,14.0,0.428571
4,260.255,3.70480,0.0,2.0,26.30,6.0,1.0,18.0,0.333333


In [25]:
print("Train/validation descriptors:", X_train_val_extended_descriptors.shape)
print("Test descriptors:", X_test_extended_descriptors.shape)

Train/validation descriptors: (7985, 9)
Test descriptors: (1997, 9)


## 4. Molecular fingerprints

Two structural fingerprint representations are generated:

### MACCS keys

MACCS keys encode the presence or absence of predefined structural patterns.

They provide a relatively compact structural representation.

### Morgan fingerprints

Morgan fingerprints encode circular atomic environments around each atom.

A radius of 2 is used, corresponding to an ECFP4-like representation, with a fixed dimensionality of 2048 bits.

In [26]:
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=MORGAN_RADIUS,
    fpSize=MORGAN_FP_SIZE,
)

In [27]:
def fingerprint_to_array(fp):
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

In [28]:
def compute_maccs_matrix(molecules):
    fingerprints = molecules.map(MACCSkeys.GenMACCSKeys)

    matrix = np.vstack(
        fingerprints
        .map(fingerprint_to_array)
        .to_numpy()
    )

    columns = [
        f"maccs_{i}"
        for i in range(matrix.shape[1])
    ]

    return pd.DataFrame(
        matrix,
        columns=columns,
    )

In [29]:
X_train_val_maccs = compute_maccs_matrix(
    train_val["ROMol"]
)

X_test_maccs = compute_maccs_matrix(
    test["ROMol"]
)

In [30]:
print("Train/validation MACCS:", X_train_val_maccs.shape)
print("Test MACCS:", X_test_maccs.shape)

Train/validation MACCS: (7985, 167)
Test MACCS: (1997, 167)


In [31]:
def compute_morgan_matrix(molecules):
    fingerprints = molecules.map(
        morgan_generator.GetFingerprint
    )

    matrix = np.vstack(
        fingerprints
        .map(fingerprint_to_array)
        .to_numpy()
    )

    columns = [
        f"morgan_{i}"
        for i in range(matrix.shape[1])
    ]

    return pd.DataFrame(
        matrix,
        columns=columns,
    )

In [32]:
X_train_val_morgan = compute_morgan_matrix(
    train_val["ROMol"]
)

X_test_morgan = compute_morgan_matrix(
    test["ROMol"]
)

In [33]:
print("Train/validation Morgan:", X_train_val_morgan.shape)
print("Test Morgan:", X_test_morgan.shape)

Train/validation Morgan: (7985, 2048)
Test Morgan: (1997, 2048)


## 5. Combined molecular representations

Physicochemical descriptors capture global molecular properties, whereas fingerprints encode structural information.

Six combined representations are therefore constructed: (We have three descriptors representations)

- Descriptors + MACCS
- Descriptors + Morgan

This will allow us to investigate whether combining global physicochemical information with structural fingerprints improves predictive performance.

In [42]:
X_train_val_descRO5_maccs = pd.concat(
    [
        X_train_val_RO5_descriptors,
        X_train_val_maccs,
    ],
    axis=1,
)

X_test_descRO5_maccs = pd.concat(
    [
        X_test_RO5_descriptors,
        X_test_maccs,
    ],
    axis=1,
)

In [35]:
X_train_val_descESOL_maccs = pd.concat(
    [
        X_train_val_esol_descriptors,
        X_train_val_maccs,
    ],
    axis=1,
)

X_test_descESOL_maccs = pd.concat(
    [
        X_test_esol_descriptors,
        X_test_maccs,
    ],
    axis=1,
)

In [36]:

X_train_val_descEXT_maccs = pd.concat(
    [
        X_train_val_extended_descriptors,
        X_train_val_maccs,
    ],
    axis=1,
) 

X_test_descEXT_maccs = pd.concat(
    [
        X_test_extended_descriptors,
        X_test_maccs,
    ],
    axis=1,
)

In [37]:
X_train_val_descRO5_morgan = pd.concat(
    [
        X_train_val_RO5_descriptors,
        X_train_val_morgan,
    ],
    axis=1,
)

X_test_descRO5_morgan = pd.concat(
    [
        X_test_RO5_descriptors,
        X_test_morgan,
    ],
    axis=1,
)

In [38]:
X_train_val_descESOL_morgan = pd.concat(
    [
        X_train_val_esol_descriptors,
        X_train_val_morgan,
    ],
    axis=1,
) 

X_test_descESOL_morgan = pd.concat(
    [
        X_test_esol_descriptors,
        X_test_morgan,
    ],
    axis=1,
)

In [39]:

X_train_val_descEXT_morgan = pd.concat(
    [
        X_train_val_extended_descriptors,
        X_train_val_morgan,
    ],
    axis=1,
) 

X_test_descEXT_morgan = pd.concat(
    [
        X_test_extended_descriptors,
        X_test_morgan,
    ],
    axis=1,
)

## 6. Target and molecular metadata

The target variable and original SMILES strings are stored separately from the feature matrices.

Keeping molecular identifiers separate from numerical features prevents accidental use of identifiers as model inputs while preserving the information required for later error analysis, molecular similarity and visualization.

In [40]:
y_train_val = (
    train_val["Y"]
    .reset_index(drop=True)
    .rename("Y")
)

y_test = (
    test["Y"]
    .reset_index(drop=True)
    .rename("Y")
)

In [41]:
metadata_train_val = (
    train_val[["Drug"]]
    .reset_index(drop=True)
)

metadata_test = (
    test[["Drug"]]
    .reset_index(drop=True)
)

## 7. Feature representation summary

The dimensions of each generated feature space are inspected before serialization.

In [43]:
feature_sets = {
    "Descriptors RO5": X_train_val_RO5_descriptors,
    "Descriptors ESOL": X_train_val_esol_descriptors,
    "Descriptors Extended": X_train_val_extended_descriptors,
    "MACCS": X_train_val_maccs,
    "Morgan": X_train_val_morgan,
    "DescriptorsRO5 + MACCS": X_train_val_descRO5_maccs,
    "DescriptorsESOL + MACCS": X_train_val_descESOL_maccs,
    "DescriptorsExtended + MACCS": X_train_val_descEXT_maccs,
    "DescriptorsRO5 + Morgan": X_train_val_descRO5_morgan,
    "DescriptorsESOL + Morgan": X_train_val_descESOL_morgan,
    "DescriptorsExtended + Morgan": X_train_val_descEXT_morgan,

}

for name, X in feature_sets.items():
    print(
        f"{name:<25} "
        f"samples={X.shape[0]:>5}, "
        f"features={X.shape[1]:>4}"
    )

Descriptors RO5           samples= 7985, features=   4
Descriptors ESOL          samples= 7985, features=   4
Descriptors Extended      samples= 7985, features=   9
MACCS                     samples= 7985, features= 167
Morgan                    samples= 7985, features=2048
DescriptorsRO5 + MACCS    samples= 7985, features= 171
DescriptorsESOL + MACCS   samples= 7985, features= 171
DescriptorsExtended + MACCS samples= 7985, features= 176
DescriptorsRO5 + Morgan   samples= 7985, features=2052
DescriptorsESOL + Morgan  samples= 7985, features=2052
DescriptorsExtended + Morgan samples= 7985, features=2057


In [44]:
n_train_val = len(train_val)
n_test = len(test)

train_val_feature_sets = {
    "descriptorsRO5": X_train_val_RO5_descriptors,
    "descriptorsESOL": X_train_val_esol_descriptors,
    "descriptorsExtended": X_train_val_extended_descriptors,
    "maccs": X_train_val_maccs,
    "morgan": X_train_val_morgan,
    "desRO5_maccs": X_train_val_descRO5_maccs,
    "desESOL_maccs": X_train_val_descESOL_maccs,
    "desExtended_maccs": X_train_val_descEXT_maccs,
    "desRO5_morgan": X_train_val_descRO5_morgan,
    "desESOL_morgan": X_train_val_descESOL_morgan,
    "desExtended_morgan": X_train_val_descEXT_morgan,
}

test_feature_sets = {
    "descriptorsRO5": X_test_RO5_descriptors,
    "descriptorsESOL": X_test_esol_descriptors,
    "descriptorsExtended": X_test_extended_descriptors,
    "maccs": X_test_maccs,
    "morgan": X_test_morgan,
    "descRO5_maccs": X_test_descRO5_maccs,
    "descESOL_maccs": X_test_descESOL_maccs,
    "descExtended_maccs": X_test_descEXT_maccs,
    "descRO5_morgan": X_test_descRO5_morgan,
    "descESOL_morgan": X_test_descESOL_morgan,
    "descExtended_morgan": X_test_descEXT_morgan,
}


assert all(
    len(X) == n_train_val
    for X in train_val_feature_sets.values()
)

assert all(
    len(X) == n_test
    for X in test_feature_sets.values()
)

assert len(y_train_val) == n_train_val
assert len(y_test) == n_test

print("Feature matrices and targets are correctly aligned.")

Feature matrices and targets are correctly aligned.


## 8. Save processed feature matrices

The generated numerical representations are stored under `data/processed/aqsoldb/`.

Parquet is used instead of CSV because it preserves numerical data types and provides a more compact representation, particularly for high-dimensional binary fingerprints.

The modelling stage can therefore load numerical feature matrices directly without depending on RDKit.

In [46]:
X_train_val_RO5_descriptors.to_parquet(
    PROCESSED_DIR / "train_val_descriptors_RO5.parquet",
    index=False,
)

X_test_RO5_descriptors.to_parquet(
    PROCESSED_DIR / "test_descriptors_RO5.parquet",
    index=False,
)

In [47]:
X_train_val_esol_descriptors.to_parquet(
    PROCESSED_DIR / "train_val_descriptors_esol.parquet",
    index=False,
)

X_test_esol_descriptors.to_parquet(
    PROCESSED_DIR / "test_descriptors_esol.parquet",
    index=False,
)

In [48]:
X_train_val_extended_descriptors.to_parquet(
    PROCESSED_DIR / "train_val_descriptors_extended.parquet",
    index=False,
)

X_test_extended_descriptors.to_parquet(
    PROCESSED_DIR / "test_descriptors_extended.parquet",
    index=False,
)

In [49]:
X_train_val_maccs.to_parquet(
    PROCESSED_DIR / "train_val_maccs.parquet",
    index=False,
)

X_test_maccs.to_parquet(
    PROCESSED_DIR / "test_maccs.parquet",
    index=False,
)

In [50]:
X_train_val_morgan.to_parquet(
    PROCESSED_DIR / "train_val_morgan.parquet",
    index=False,
)

X_test_morgan.to_parquet(
    PROCESSED_DIR / "test_morgan.parquet",
    index=False,
)

In [51]:
X_train_val_descRO5_maccs.to_parquet(
    PROCESSED_DIR / "train_val_descRO5_maccs.parquet",
    index=False,
)

X_test_descRO5_maccs.to_parquet(
    PROCESSED_DIR / "test_descRO5_maccs.parquet",
    index=False,
)

In [52]:
X_train_val_descESOL_maccs.to_parquet(
    PROCESSED_DIR / "train_val_descESOL_maccs.parquet",
    index=False,
)

X_test_descESOL_maccs.to_parquet(
    PROCESSED_DIR / "test_descESOL_maccs.parquet",
    index=False,
)

In [53]:
X_train_val_descEXT_maccs.to_parquet(
    PROCESSED_DIR / "train_val_descEXT_maccs.parquet",
    index=False,
)

X_test_descEXT_maccs.to_parquet(
    PROCESSED_DIR / "test_descEXT_maccs.parquet",
    index=False,
)

In [54]:
X_train_val_descRO5_morgan.to_parquet(
    PROCESSED_DIR / "train_val_descRO5_morgan.parquet",
    index=False,
)

X_test_descRO5_morgan.to_parquet(
    PROCESSED_DIR / "test_descRO5_morgan.parquet",
    index=False,
)

In [55]:
X_train_val_descESOL_morgan.to_parquet(
    PROCESSED_DIR / "train_val_descESOL_morgan.parquet",
    index=False,
)

X_test_descESOL_morgan.to_parquet(
    PROCESSED_DIR / "test_descESOL_morgan.parquet",
    index=False,
)

In [56]:
X_train_val_descEXT_morgan.to_parquet(
    PROCESSED_DIR / "train_val_descEXT_morgan.parquet",
    index=False,
)

X_test_descEXT_morgan.to_parquet(
    PROCESSED_DIR / "test_descEXT_morgan.parquet",
    index=False,
)

In [57]:
y_train_val.to_frame().to_parquet(
    PROCESSED_DIR / "train_val_target.parquet",
    index=False,
)

y_test.to_frame().to_parquet(
    PROCESSED_DIR / "test_target.parquet",
    index=False,
)

In [58]:
metadata_train_val.to_csv(
    PROCESSED_DIR / "train_val_metadata.csv",
    index=False,
)

metadata_test.to_csv(
    PROCESSED_DIR / "test_metadata.csv",
    index=False,
)

In [59]:
saved_files = sorted(
    list(PROCESSED_DIR.glob("*.parquet"))
    + list(PROCESSED_DIR.glob("*.csv"))
)

for file in saved_files:
    print(file.name)

test_descESOL_maccs.parquet
test_descESOL_morgan.parquet
test_descEXT_maccs.parquet
test_descEXT_morgan.parquet
test_descriptors_esol.parquet
test_descriptors_extended.parquet
test_descriptors_RO5.parquet
test_descRO5_maccs.parquet
test_descRO5_morgan.parquet
test_maccs.parquet
test_metadata.csv
test_morgan.parquet
test_target.parquet
train_val_descESOL_maccs.parquet
train_val_descESOL_morgan.parquet
train_val_descEXT_maccs.parquet
train_val_descEXT_morgan.parquet
train_val_descriptors_esol.parquet
train_val_descriptors_extended.parquet
train_val_descriptors_RO5.parquet
train_val_descRO5_maccs.parquet
train_val_descRO5_morgan.parquet
train_val_maccs.parquet
train_val_metadata.csv
train_val_morgan.parquet
train_val_target.parquet


## Conclusions

Five molecular feature representations were generated for the AqSolDB benchmark:

| Representation | Description |
|---|---|
| Descriptors | Global physicochemical molecular properties |
| MACCS | Predefined structural keys |
| Morgan | Circular structural fingerprints |
| Descriptors + MACCS | Physicochemical + predefined structural information |
| Descriptors + Morgan | Physicochemical + circular structural information |

The same deterministic transformations were applied independently to the official TDC training/validation and test partitions.

The resulting numerical matrices were stored in `data/processed/aqsoldb/` and can now be used directly during model development.

No scaling, feature selection or model-dependent preprocessing has been applied. These operations will be performed inside the modelling pipeline using training data only in order to avoid data leakage.

The next stage will compare machine-learning models across these molecular representations using the official TDC scaffold-aware benchmark protocol.